# Bryan, TX — LVT Policy Analysis

Models a revenue-neutral split-rate LVT shift for parcels inside the
**City of Bryan** corporate limits (Brazos County, TX).

## Policy decisions (from PR #8)

| Choice | Setting | Note |
|---|---|---|
| Geographic scope | City of Bryan via spatial filter | Brazos CAD shapefile covers the whole county |
| Reform shape | Split-rate at 2:1 | Primary scenario |
| Tax stack | City of Bryan + Brazos County + Bryan ISD | Three-entity full-stack tax math |
| 2025 adopted rates | Bryan 0.6240, BC 0.4197, ISD 0.9469 per $100 | BCAD published rates |
| Homestead exemptions | Bryan 20% of appraised; ISD $40K flat; BC $0 | Texas Tax Code |
| Land/improvement values | Shapefile `Land_Val` / `Imprv_Val` | BCAD certified |

Lars's PR #8 used the BCAD PACS fixed-width `APPRAISAL_INFO.TXT` parsing.
This port simplifies by using the shapefile's pre-extracted `Land_Val`,
`Imprv_Val`, `state_cd`, and `Exemptions` columns directly — they're already
parsed and joined in the BCAD shapefile.


## Section 1 — Imports and setup

In [ ]:
import sys
import io
import os
import time
import zipfile
import tempfile
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from dotenv import load_dotenv
from shapely.geometry import Polygon

sys.path.insert(0, '../..')
REPO_ROOT = Path('../..').resolve()
load_dotenv(REPO_ROOT / '.env')

from lvt.lvt_utils import (
    model_split_rate_tax,
    calculate_category_tax_summary,
    print_category_tax_summary,
    save_standard_export,
)
from lvt.census_utils import get_census_data_with_boundaries, match_to_census_blockgroups

CITY_NAME = 'bryan'
STATE_FIPS = '48'
COUNTY_FIPS = '041'  # Brazos County
MODEL_TYPE = 'split_rate_2to1'
LAND_IMPROVEMENT_RATIO = 2.0

DATA_DIR = Path('data')
DATA_DIR.mkdir(parents=True, exist_ok=True)

SHP_URL = 'https://brazoscad.org/wp-content/uploads/2025/11/Public_Parcel_Boundary_certified.zip'
BRYAN_BOUNDARY_URL = ('https://services.arcgis.com/7kZ9ejBkXzDTCStF/ArcGIS/rest/'
                      'services/Bryan_City_Boundary/FeatureServer/8/query')

# 2025 adopted tax rates (per $100 of assessed value)
MILLAGE_BRYAN = 0.6240
MILLAGE_BC = 0.4197
MILLAGE_BRYANISD = 0.9469

# Entity-specific exemptions (2025)
BRYAN_HOMESTEAD_RATE = 0.20
BRYAN_OV65_AMT = 0
BC_HOMESTEAD_AMT = 0
BC_OV65_AMT = 0
BRYANISD_HOMESTEAD_AMT = 40_000
BRYANISD_OV65_AMT = 10_000

## Section 2 — Fetch BCAD parcel shapefile, spatial filter to Bryan

In [ ]:
def fetch_arcgis_geojson(url):
    session = requests.Session()
    count = int(session.get(url, params={'f':'json','where':'1=1','returnCountOnly':'true'},
                            timeout=60).json().get('count', 0))
    feats = []
    for off in range(0, count, 2000):
        r = session.get(url, params={
            'f':'json','where':'1=1','outFields':'*','returnGeometry':'true',
            'resultOffset':off,'resultRecordCount':2000,'outSR':4326,
        }, timeout=180)
        feats.extend(r.json().get('features', []))
    rows = []
    for f in feats:
        attrs = f.get('attributes', {})
        g = f.get('geometry', {}) or {}
        rings = g.get('rings')
        if rings:
            attrs['geometry'] = Polygon(rings[0], holes=rings[1:] if len(rings) > 1 else None)
            rows.append(attrs)
    return gpd.GeoDataFrame(rows, geometry='geometry', crs='EPSG:4326')


def load_city_boundary():
    cache = DATA_DIR / 'bryan_city_boundary.parquet'
    if cache.exists():
        gdf_b = gpd.read_parquet(cache)
    else:
        print('Downloading Bryan city boundary...')
        gdf_b = fetch_arcgis_geojson(BRYAN_BOUNDARY_URL)
        gdf_b.to_parquet(cache, index=False)
    return gdf_b.to_crs(epsg=3857).union_all()


def _pick_parcel_shp(shp_files):
    keywords_avoid = {'ABSTRACT','SUBDIV','ADDRESS','ID','XREF','HOOK','GRID','TEXT',
                      'NUMBER','NAME','LINE','CREEK','LAKE','POND','ROAD','RAILROAD',
                      'UDI','PHASE','BLOCK','LOT','CENTER'}
    def score(p):
        fn = p.split('/')[-1].upper().replace('.SHP', '')
        parts = set(fn.split('_'))
        if fn in ('PUBLIC_PARCEL_BOUNDARY_CERTIFIED', 'PUBLIC_PARCEL_BOUNDARY', 'PARCEL', 'PARCELS'):
            return 10
        if 'PARCEL' in fn and not (parts & keywords_avoid):
            return 5
        if 'PARCEL' in fn:
            return 2
        return 0
    return sorted(shp_files, key=score, reverse=True)[0]


def load_bryan_parcels():
    cache = DATA_DIR / 'bryan_parcels.parquet'
    if cache.exists():
        return gpd.read_parquet(cache)
    print(f'Downloading BCAD parcel shapefile (~72MB) from {SHP_URL}...')
    resp = requests.get(SHP_URL, stream=True, timeout=600)
    resp.raise_for_status()
    buf = io.BytesIO()
    for chunk in resp.iter_content(256 * 1024):
        buf.write(chunk)
    buf.seek(0)
    print(f'  downloaded {buf.tell()/1e6:.1f} MB')
    with zipfile.ZipFile(buf) as zf:
        shps = [n for n in zf.namelist() if n.lower().endswith('.shp')]
        chosen = _pick_parcel_shp(shps)
        print(f'  using {chosen}')
        with tempfile.TemporaryDirectory() as tmpdir:
            zf.extractall(tmpdir)
            gdf_all = gpd.read_file(Path(tmpdir) / chosen)
    print(f'  total county parcels: {len(gdf_all):,}')

    boundary_poly = load_city_boundary()
    gdf_3857 = gdf_all.to_crs(epsg=3857)
    mask = gdf_3857.geometry.centroid.within(boundary_poly)
    gdf_city = gdf_all[mask].copy()
    print(f'  inside Bryan city limits: {len(gdf_city):,}')
    gdf_city.to_parquet(cache, index=False)
    return gdf_city


t0 = time.time()
gdf = load_bryan_parcels()
print(f'Loaded {len(gdf):,} Bryan parcels in {time.time()-t0:.1f}s')

# Coerce numerics
for col in ['Land_Val', 'Imprv_Val', 'market', 'land_acres']:
    if col in gdf.columns:
        gdf[col] = pd.to_numeric(gdf[col], errors='coerce').fillna(0)

## Section 3 — Property classification + exemption flags

In [ ]:
# Parse Exemptions string (e.g. "HS, OV65", "EX-XV", "DV4, HS")
gdf['Exemptions'] = gdf['Exemptions'].fillna('').astype(str)
gdf['hs_exempt'] = gdf['Exemptions'].str.contains(r'\bHS\b', regex=True, na=False)
gdf['ov65_exempt'] = gdf['Exemptions'].str.contains(r'\bOV65\b', regex=True, na=False)
gdf['full_exmp_flag'] = (
    gdf['Exemptions'].str.contains(r'\bEX[-\b]', regex=True, na=False)
    | gdf['Exemptions'].str.contains(r'\bEX$', regex=True, na=False)
)

# Texas Property Tax Assistance Division (PTAD) state-code classification.
# Splits A* and B* subcodes into single-family / multi-family / mobile-home /
# condo-townhome, etc., rather than collapsing all "residential" into one bucket.
def categorize(state_cd):
    code = str(state_cd or '').strip().upper()
    if code in ('A1', 'A3'):
        return 'Single Family Residential'
    if code == 'A2':
        return 'Mobile Home'
    if code in ('A7', 'A8'):
        return 'Townhome / Rowhouse'
    if code == 'A9':
        return 'Condominium'
    if code.startswith('A'):
        return 'Other Residential'
    if code == 'B1':
        return 'Large Multi-Family (5+ units)'
    if code in ('B2', 'B3', 'B4'):
        return 'Small Multi-Family (2-4 units)'
    if code in ('C1', 'C2'):
        return 'Vacant / Undeveloped'
    if code in ('D1', 'D2'):
        return 'Agricultural'
    if code == 'E1':
        return 'Single Family Residential'  # rural single residence
    if code in ('E2', 'E3', 'E4'):
        return 'Rural / Acreage'
    if code == 'F2':
        return 'Industrial'
    if code.startswith('F'):
        return 'Commercial'
    if code.startswith(('G', 'J', 'X')):
        return 'Exempt / Government'
    if code == 'O1':
        return 'Vacant / Undeveloped'
    if code == 'O2':
        return 'Other Residential'
    return 'Other'


gdf['PROPERTY_CATEGORY'] = gdf['state_cd'].apply(categorize)
gdf['full_exmp'] = (
    gdf['full_exmp_flag']
    | (gdf['PROPERTY_CATEGORY'] == 'Exempt / Government')
).astype(int)

print(gdf['PROPERTY_CATEGORY'].value_counts())
print(f'\nFully exempt: {gdf["full_exmp"].sum():,}')
print(f'Homestead (HS): {gdf["hs_exempt"].sum():,}')
print(f'Over-65 (OV65): {gdf["ov65_exempt"].sum():,}')

## Section 4 — Three-entity current tax (City + County + Bryan ISD)

In [ ]:
# Use market value as appraised proxy (BCAD shapefile doesn't include assessed_val
# separately; market value = appraised value for most parcels)
gdf['appraised_val'] = gdf['market']
gdf['assessed_val'] = gdf['market']  # appraised == assessed unless 10% cap applies (homesteads only)

# City of Bryan: 20% homestead off appraised value
city_hs = np.where(gdf['hs_exempt'], BRYAN_HOMESTEAD_RATE * gdf['appraised_val'], 0)
city_o65 = np.where(gdf['ov65_exempt'], BRYAN_OV65_AMT, 0)
gdf['city_taxable'] = np.where(
    gdf['full_exmp'] == 1, 0,
    np.maximum(0, gdf['assessed_val'] - city_hs - city_o65),
)

# Brazos County: no optional exemptions in our model
bc_hs = np.where(gdf['hs_exempt'], BC_HOMESTEAD_AMT, 0)
bc_o65 = np.where(gdf['ov65_exempt'], BC_OV65_AMT, 0)
gdf['bc_taxable'] = np.where(
    gdf['full_exmp'] == 1, 0,
    np.maximum(0, gdf['assessed_val'] - bc_hs - bc_o65),
)

# Bryan ISD: $40K mandatory homestead + $10K additional over-65
isd_hs = np.where(gdf['hs_exempt'], BRYANISD_HOMESTEAD_AMT, 0)
isd_o65 = np.where(gdf['ov65_exempt'], BRYANISD_OV65_AMT, 0)
gdf['isd_taxable'] = np.where(
    gdf['full_exmp'] == 1, 0,
    np.maximum(0, gdf['assessed_val'] - isd_hs - isd_o65),
)

gdf['current_tax_city'] = gdf['city_taxable'] * MILLAGE_BRYAN / 100
gdf['current_tax_bc'] = gdf['bc_taxable'] * MILLAGE_BC / 100
gdf['current_tax_isd'] = gdf['isd_taxable'] * MILLAGE_BRYANISD / 100
gdf['current_tax'] = gdf['current_tax_city'] + gdf['current_tax_bc'] + gdf['current_tax_isd']

print('Combined three-entity current tax:')
print(f'  City of Bryan: ${gdf["current_tax_city"].sum():>15,.0f}')
print(f'  Brazos County: ${gdf["current_tax_bc"].sum():>15,.0f}')
print(f'  Bryan ISD:     ${gdf["current_tax_isd"].sum():>15,.0f}')
print(f'  ─────────────────────────────')
print(f'  Total:         ${gdf["current_tax"].sum():>15,.0f}')

# --- Exemption-preserving scaling for the split-rate solver ---------------
# The split-rate solver expects current_tax = (taxable_land + taxable_imp) * uniform_mill / 1000.
# Bryan's current_tax embeds per-parcel HS / OV65 exemptions across three
# entities — passing raw Land_Val / Imprv_Val to the solver would silently
# strip those exemptions from the LVT result (homesteaders would lose their
# 20% city HS + $40K ISD HS in the new tax math, jacking up residential).
# Scale per-parcel land/improvement values so that under a single uniform
# combined-entity rate they reproduce the actual current_tax. The LVT shift
# then operates on already-exemption-adjusted bases and behaves as expected.
COMBINED_RATE_PER_DOLLAR = (MILLAGE_BRYAN + MILLAGE_BC + MILLAGE_BRYANISD) / 100.0  # per $1 of value
raw_total = gdf['Land_Val'].clip(lower=0) + gdf['Imprv_Val'].clip(lower=0)
no_exempt_tax = raw_total * COMBINED_RATE_PER_DOLLAR
# parcel-level scaling factor; falls back to 1 when raw_total is zero
scale = np.where(no_exempt_tax > 0, gdf['current_tax'] / no_exempt_tax, 0.0)
gdf['taxable_land_value'] = gdf['Land_Val'].clip(lower=0) * scale
gdf['taxable_improvement_value'] = gdf['Imprv_Val'].clip(lower=0) * scale
current_revenue = float(gdf['current_tax'].sum())
print(f'\nExemption-scaling diagnostics:')
print(f'  median scale: {np.median(scale[scale>0]):.3f}  (1.0 = no HS, lower = more exempted)')
print(f'  raw taxable base:    ${raw_total.sum():>15,.0f}')
print(f'  scaled taxable base: ${(gdf["taxable_land_value"] + gdf["taxable_improvement_value"]).sum():>15,.0f}')

## Section 5 — Split-rate model (2:1)

In [ ]:
solve_df = gdf[gdf['full_exmp'] == 0].copy()
print(f'Solving on {len(solve_df):,} parcels (excluded fully exempt: {len(gdf) - len(solve_df):,})')

land_millage, improvement_millage, new_revenue, solve_df = model_split_rate_tax(
    df=solve_df,
    land_value_col='taxable_land_value',
    improvement_value_col='taxable_improvement_value',
    current_revenue=float(solve_df['current_tax'].sum()),
    land_improvement_ratio=LAND_IMPROVEMENT_RATIO,
)

excluded = gdf[gdf['full_exmp'] == 1].copy()
excluded['new_tax'] = 0.0
excluded['tax_change'] = 0.0
excluded['tax_change_pct'] = 0.0
gdf = pd.concat([solve_df, excluded]).sort_index()

print(f'\nLand millage (rate per $1000):        {land_millage:.4f}')
print(f'Improvement millage (rate per $1000): {improvement_millage:.4f}')
print(f'Revenue: ${new_revenue:,.0f} (target ${current_revenue:,.0f})')

In [ ]:
category_summary = calculate_category_tax_summary(
    df=gdf, category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax', new_tax_col='new_tax',
)
print_category_tax_summary(category_summary, title='Bryan — 2:1 Split-Rate Tax Impact')

## Section 6 — Exploration chart

In [ ]:
import matplotlib
matplotlib.use('Agg')
fig, ax = plt.subplots(figsize=(10, 5))
summary = gdf.groupby('PROPERTY_CATEGORY')['tax_change_pct'].median().sort_values()
summary.plot.barh(ax=ax, color=np.where(summary < 0, '#228B22', '#8B0000'))
ax.axvline(0, color='black', linewidth=1)
ax.set_title('Bryan — Median Tax Change % by Category (2:1 Split-Rate)')
ax.set_xlabel('Median % Change')
plt.tight_layout()
plt.savefig(DATA_DIR / 'category_preview.png', dpi=120)
plt.close()
print('Saved preview chart')

## Section 7 — Census join + export

In [ ]:
import concurrent.futures

_fips = STATE_FIPS + COUNTY_FIPS
try:
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as _ex:
        _future = _ex.submit(get_census_data_with_boundaries, _fips, 2022)
        try:
            census_data, census_gdf = _future.result(timeout=90)
            gdf = match_to_census_blockgroups(gdf, census_gdf)
            if 'minority_pct' not in gdf.columns and 'total_pop' in gdf.columns and 'white_pop' in gdf.columns:
                gdf['minority_pct'] = ((gdf['total_pop'] - gdf['white_pop']) / gdf['total_pop'] * 100).round(2)
            if 'black_pct' not in gdf.columns and 'total_pop' in gdf.columns and 'black_pop' in gdf.columns:
                gdf['black_pct'] = (gdf['black_pop'] / gdf['total_pop'] * 100).round(2)
            print(f"Census join: {gdf['std_geoid'].notna().mean()*100:.1f}% matched")
        except concurrent.futures.TimeoutError:
            print('Census API timed out — skipping census join')
            for _col in ['std_geoid', 'median_income', 'minority_pct', 'black_pct']:
                gdf[_col] = float('nan')
except Exception as e:
    print(f'Census join failed: {e}')
    for _col in ['std_geoid', 'median_income', 'minority_pct', 'black_pct']:
        gdf[_col] = float('nan')

In [ ]:
out_df = save_standard_export(
    df=gdf,
    city=CITY_NAME,
    output_path=f'../../analysis/data/{CITY_NAME}.csv',
    model_type=MODEL_TYPE,
    land_millage=land_millage,
    improvement_millage=improvement_millage,
    property_category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
    tax_change_col='tax_change',
    tax_change_pct_col='tax_change_pct',
    taxable_land_col='taxable_land_value',
    taxable_improvement_col='taxable_improvement_value',
)
print('Done.')